# Интеллектуальный почтовый помощник (EmailSmartAssistant)В этой записной книжке реализована полная интеллектуальная система обработки почты, в том числе:-Автоматическая сортировка сообщений-Генерация черновика Smart Response-Умные напоминания о том, что важно-Извлечение информации о ключе сообщения-Архивирование и организация почты

## 1. Импорт необходимых библиотек

In [ ]:
import imaplib
import smtplib
import email
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from email.header import decode_header
import json
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import re
import jieba
from textblob import TextBlob
from langdetect import detect
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
import dateparser
import arrow
from jinja2 import Template
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from rich.console import Console
from rich.table import Table
from rich.panel import Panel
import warnings
warnings.filterwarnings('ignore')

# Установить китайский шрифтplt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

console = Console()
print("✅ Все библиотеки успешно импортированы!")

## 2. Загрузка конфигурации

In [ ]:
# Профиль нагрузкиdef load_config():
    try:
        with open('config/email_config.json', 'r', encoding='utf-8') as f:
            config = json.load(f)
        console.print("✅Конфигурационный файл успешно загружен", style="green")
        return config
    except FileNotFoundError:
        console.print("❌Конфигурационный файл не найден, проверьте конфигурацию/email_config.json", style="red")
        return None

# Загрузить шаблон ответаdef load_templates():
    try:
        with open('templates/reply_templates.json', 'r', encoding='utf-8') as f:
            templates = json.load(f)
        console.print("✅Шаблон ответа успешно загружен", style="green")
        return templates
    except FileNotFoundError:
        console.print("❌Файл шаблона не найден, проверьте шаблоны/reply_templates.json", style="red")
        return None

config = load_config()
templates = load_templates()

## 3. Класс подключения и получения почты

In [ ]:
class EmailConnector:
    def __init__(self, email_config):
        self.config = email_config
        self.imap_conn = None
        self.smtp_conn = None
    
    def connect_imap(self):
        """Подключение к IMAP-серверу"""
        try:
            self.imap_conn = imaplib.IMAP4_SSL(self.config['imap_server'], self.config['imap_port'])
            self.imap_conn.login(self.config['email'], self.config['password'])
            console.print(f"✅Соединение IMAP успешно: {self.config['email']}", style="green")
            return True
        except Exception as e:
            console.print(f"❌Ошибка подключения к IMAP: {str(e)}", style="red")
            return False
    
    def get_emails(self, folder='INBOX', limit=50):
        """Получение списка писем"""
        if not self.imap_conn:
            if not self.connect_imap():
                return []
        
        try:
            self.imap_conn.select(folder)
            status, messages = self.imap_conn.search(None, 'ALL')
            
            if status != 'OK':
                return []
            
            email_ids = messages[0].split()
            # Получайте последние сообщения            email_ids = email_ids[-limit:] if len(email_ids) > limit else email_ids
            
            emails = []
            for email_id in tqdm(email_ids, desc="Получить почту"):
                status, msg_data = self.imap_conn.fetch(email_id, '(RFC822)')
                if status == 'OK':
                    email_message = email.message_from_bytes(msg_data[0][1])
                    emails.append(self.parse_email(email_message, email_id.decode()))
            
            return emails
        except Exception as e:
            console.print(f"❌Не удалось получить сообщения: {str(e)}", style="red")
            return []
    
    def parse_email(self, email_message, email_id):
        """Разбор содержимого письма"""
        # Расшифровать заголовок письма        def decode_mime_words(s):
            return ''.join(
                word.decode(encoding or 'utf-8') if isinstance(word, bytes) else word
                for word, encoding in decode_header(s)
            )
        
        subject = decode_mime_words(email_message['Subject'] or '')
        sender = decode_mime_words(email_message['From'] or '')
        date = email_message['Date']
        
        # Получить текст сообщения        body = ""
        if email_message.is_multipart():
            for part in email_message.walk():
                if part.get_content_type() == "text/plain":
                    try:
                        body = part.get_payload(decode=True).decode('utf-8')
                        break
                    except:
                        continue
        else:
            try:
                body = email_message.get_payload(decode=True).decode('utf-8')
            except:
                body = str(email_message.get_payload())
        
        return {
            'id': email_id,
            'subject': subject,
            'sender': sender,
            'date': date,
            'body': body,
            'raw_message': email_message
        }
    
    def close_connections(self):
        """Закрытие соединения"""
        if self.imap_conn:
            self.imap_conn.close()
            self.imap_conn.logout()
        if self.smtp_conn:
            self.smtp_conn.quit()

print("✅Определение класса почтового соединителя завершено")

## 4. Классификатор почты

In [ ]:
class EmailClassifier:
    def __init__(self, config):
        self.config = config
        self.classification_rules = config['classification_rules']
        self.priority_rules = config['priority_rules']
    
    def classify_email_type(self, email_data):
        """Классификация типа письма"""
        subject = email_data['subject'].lower()
        body = email_data['body'].lower()
        sender = email_data['sender'].lower()
        
        text_content = f"{subject} {body}"
        
        # Проверка спам-ключевых слов        spam_score = sum(1 for keyword in self.classification_rules['spam_keywords'] 
                        if keyword in text_content)
        if spam_score >= 2:
            return 'spam'
        
        # Проверить ключевые слова рабочей электронной почты        work_score = sum(1 for keyword in self.classification_rules['work_keywords'] 
                        if keyword in text_content)
        
        # Проверьте ключевые слова запроса клиента        customer_score = sum(1 for keyword in self.classification_rules['customer_keywords'] 
                           if keyword in text_content)
        
        # Проверить личные ключевые слова электронной почты        personal_score = sum(1 for keyword in self.classification_rules['personal_keywords'] 
                           if keyword in text_content)
        
        # Определить тип на основе оценки        scores = {
            'work': work_score,
            'customer': customer_score,
            'personal': personal_score
        }
        
        return max(scores, key=scores.get) if max(scores.values()) > 0 else 'other'
    
    def classify_priority(self, email_data):
        """Классификация приоритета письма"""
        subject = email_data['subject'].lower()
        body = email_data['body'].lower()
        sender = email_data['sender']
        
        text_content = f"{subject} {body}"
        
        # Проверка высокоприоритетных отправителей        if any(priority_sender in sender for priority_sender in self.priority_rules['high_priority_senders']):
            return 'high'
        
        # Проверка на наличие высокоприоритетных ключевых слов        high_priority_score = sum(1 for keyword in self.priority_rules['high_priority_keywords'] 
                                 if keyword in text_content)
        if high_priority_score > 0:
            return 'high'
        
        # Проверить низкоприоритетные ключевые слова        low_priority_score = sum(1 for keyword in self.priority_rules['low_priority_keywords'] 
                                if keyword in text_content)
        if low_priority_score > 0:
            return 'low'
        
        return 'medium'
    
    def classify_sender_type(self, email_data):
        """Классификация типа отправителя"""
        sender = email_data['sender'].lower()
        
        # Простая логика классификации отправителей        if any(domain in sender for domain in ['@company.com', '@work.com']):
            return 'colleague'
        elif 'noreply' in sender or 'no-reply' in sender:
            return 'system'
        elif any(keyword in sender for keyword in ['service', 'support', 'info']):
            return 'customer_service'
        else:
            return 'external'
    
    def classify_email(self, email_data):
        """Полная классификация письма"""
        return {
            'type': self.classify_email_type(email_data),
            'priority': self.classify_priority(email_data),
            'sender_type': self.classify_sender_type(email_data)
        }

print("✅Определение классификатора сообщений завершено")

## 5. Извлекатель ключевой информации

In [ ]:
class InformationExtractor:
    def __init__(self):
        # Регулярные выражения, связанные со временем        self.date_patterns = [
            r'\d{4}[-/]\d{1,2}[-/]\d{1,2}',  #2024-01-01 или 2024/01/01
            r'\d{1,2}[-/]\d{1,2}[-/]\d{4}',  #01-01-2024 или 01/01/2024
            r'\d{1,2}мес.\d{1,2}дн.',           #1 январяа            r'\d{1,2}/\d{1,2}',              # 1/1
        ]
        
        # Ключевые слова, связанные со временем        self.time_keywords = [
            'По состоянию на:', 'deadline', 'В срок', 'Время завершения', 'Срок поставки',
            'Время проведения совещания', 'Запланированное время', 'ЗАЯВКА', 'Организация'
        ]
        
        # Ключевые слова To-Do        self.todo_keywords = [
            'Потребность', '请', 'Требования', 'Ок', 'Меры по', 'Подготовка',
            'need', 'please', 'require', 'complete', 'prepare'
        ]
    
    def extract_dates(self, text):
        """Извлечение дат из текста"""
        dates = []
        
        # Извлечение дат с помощью регулярных выражений        for pattern in self.date_patterns:
            matches = re.findall(pattern, text)
            dates.extend(matches)
        
        # Используйте dateparser для разрешения более сложных выражений даты        sentences = text.split('。')
        for sentence in sentences:
            if any(keyword in sentence for keyword in self.time_keywords):
                parsed_date = dateparser.parse(sentence)
                if parsed_date:
                    dates.append(parsed_date.strftime('%Y-%m-%d'))
        
        return list(set(dates))  #Дедупликация    
    def extract_todos(self, text):
        """Извлечение задач"""
        todos = []
        sentences = text.split('。')
        
        for sentence in sentences:
            if any(keyword in sentence for keyword in self.todo_keywords):
                # Очистить предложения                clean_sentence = sentence.strip()
                if len(clean_sentence) > 5:  #Отфильтровать слишком короткие предложения                    todos.append(clean_sentence)
        
        return todos
    
    def extract_contacts(self, text):
        """Извлечение контактной информации"""
        contacts = {
            'emails': [],
            'phones': []
        }
        
        # Получить адрес электронной почты        email_pattern = r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b'
        contacts['emails'] = re.findall(email_pattern, text)
        
        # Получение телефонных номеров        phone_patterns = [
            r'1[3-9]\d{9}',  #Номер телефона в Китае            r'\d{3}-\d{4}-\d{4}',  #Формат телефона            r'\(\d{3}\)\s*\d{3}-\d{4}'  #Американский формат телефона        ]
        
        for pattern in phone_patterns:
            contacts['phones'].extend(re.findall(pattern, text))
        
        return contacts
    
    def generate_summary(self, email_data):
        """Генерация краткого содержания письма"""
        subject = email_data['subject']
        body = email_data['body']
        sender = email_data['sender']
        
        # Извлечь ключевую информацию        dates = self.extract_dates(body)
        todos = self.extract_todos(body)
        contacts = self.extract_contacts(body)
        
        # Сформировать сводку        summary = {
            'subject': subject,
            'sender': sender,
            'key_dates': dates,
            'todo_items': todos[:3],  #До 3 задач            'contacts': contacts,
            'body_preview': body[:200] + '...' if len(body) > 200 else body
        }
        
        return summary

print("✅Определение экстрактора информации завершено")

## 6. Генератор интеллектуальных ответов

In [ ]:
class ReplyGenerator:
    def __init__(self, templates, config):
        self.templates = templates
        self.config = config
        self.reply_settings = config['reply_settings']
    
    def detect_language(self, text):
        """Определение языка текста"""
        try:
            lang = detect(text)
            return 'zh' if lang == 'zh-cn' else 'en'
        except:
            return 'zh'  #Китайский по умолчанию    
    def select_template(self, email_classification, email_data):
        """Выбор шаблона по классификации письма"""
        email_type = email_classification['type']
        
        # Выберите шаблон в зависимости от типа сообщения        if email_type == 'work':
            if '会议' in email_data['subject'] or 'meeting' in email_data['subject'].lower():
                return 'work_meeting'
            else:
                return 'general_acknowledgment'
        elif email_type == 'customer':
            return 'customer_inquiry'
        else:
            return 'general_acknowledgment'
    
    def generate_reply(self, email_data, email_classification):
"" "Сгенерировать черновик ответа" ""        # Выберите один из шаблонов        template_key = self.select_template(email_classification, email_data)
        
        # Определить язык        language = self.detect_language(email_data['body'])
        
        # Подтвердите тон (формальный/Неофициальные группы        tone = 'formal' if self.reply_settings['formal_tone'] else 'casual'
        
        # Получить шаблон        try:
            template_text = self.templates[template_key][tone][language]
        except KeyError:
            # Если соответствующий шаблон не найден, используйте общий шаблон подтверждения            template_text = self.templates['general_acknowledgment']['formal'][language]
        
        # Подготовка переменных шаблона        template_vars = {
            'subject': email_data['subject'],
            'timeframe': '24часов' if language == 'zh' else '24 hours',
            'return_date': (datetime.now() + timedelta(days=1)).strftime('%Y-%m-%d'),
            'emergency_contact': 'assistant@company.com'
        }
        
        # Шаблоны визуализации        template = Template(template_text)
        reply_content = template.render(**template_vars)
        
        # Сгенерировать полный ответ        reply = {
            'to': email_data['sender'],
            'subject': f"Re: {email_data['subject']}",
            'content': reply_content,
            'template_used': template_key,
            'tone': tone,
            'language': language
        }
        
        return reply

print("✅Определение генератора отклика завершено")

## 7. Менеджер напоминаний

In [ ]:
class ReminderManager:
    def __init__(self, config):
        self.config = config
        self.reminder_settings = config['reminder_settings']
        self.reminders = []
    
    def create_reminders(self, email_data, extracted_info):
        """Создание напоминаний по извлечённой информации"""
        reminders = []
        
        # Создать напоминание для каждой критической даты        for date_str in extracted_info['key_dates']:
            try:
                target_date = datetime.strptime(date_str, '%Y-%m-%d')
                
                # Создайте напоминание для каждого дня выполнения заказа                for advance_days in self.reminder_settings['advance_days']:
                    reminder_date = target_date - timedelta(days=advance_days)
                    
                    # Создавать только будущие напоминания                    if reminder_date > datetime.now():
                        reminder = {
                            'id': f"{email_data['id']}_{date_str}_{advance_days}",
                            'email_id': email_data['id'],
                            'email_subject': email_data['subject'],
                            'reminder_date': reminder_date,
                            'target_date': target_date,
                            'advance_days': advance_days,
                            'message': f"Напоминание: {email_data['subject']} — через {advance_days} дн. ({date_str})",
                            'status': 'pending'
                        }
                        reminders.append(reminder)
            except ValueError:
                continue  #Пропустить неразрешимые даты        
        # Создайте напоминание для списка дел        for todo in extracted_info['todo_items']:
            reminder = {
                'id': f"{email_data['id']}_todo_{hash(todo) % 10000}",
                'email_id': email_data['id'],
                'email_subject': email_data['subject'],
                'reminder_date': datetime.now() + timedelta(hours=2),  #Напомнить через 2 часа                'target_date': None,
                'advance_days': 0,
                'message': f"Напоминание о задаче: {todo}",
                'status': 'pending'
            }
            reminders.append(reminder)
        
        self.reminders.extend(reminders)
        return reminders
    
    def get_pending_reminders(self):
        """Получение ожидающих напоминаний"""
        now = datetime.now()
        pending = []
        
        for reminder in self.reminders:
            if (reminder['status'] == 'pending' and 
                reminder['reminder_date'] <= now):
                pending.append(reminder)
        
        return pending
    
    def mark_reminder_sent(self, reminder_id):
        """Отметка напоминания как отправленного"""
        for reminder in self.reminders:
            if reminder['id'] == reminder_id:
                reminder['status'] = 'sent'
                break
    
    def get_reminders_summary(self):
        """Получение сводки напоминаний"""
        total = len(self.reminders)
        pending = len([r for r in self.reminders if r['status'] == 'pending'])
        sent = len([r for r in self.reminders if r['status'] == 'sent'])
        
        return {
            'total': total,
            'pending': pending,
            'sent': sent
        }

print("✅Определение диспетчера напоминаний завершено")

## 8. Основная программа — интеллектуальный почтовый помощник

In [ ]:
class EmailSmartAssistant:
    def __init__(self, config, templates):
        self.config = config
        self.templates = templates
        
        # Инициализация отдельных компонентов        self.connector = None
        self.classifier = EmailClassifier(config)
        self.extractor = InformationExtractor()
        self.reply_generator = ReplyGenerator(templates, config)
        self.reminder_manager = ReminderManager(config)
        
        # Хранение результатов процесса        self.processed_emails = []
        self.processing_stats = {
            'total_emails': 0,
            'classified_emails': 0,
            'replies_generated': 0,
            'reminders_created': 0
        }
    
    def connect_email_account(self, account_index=0):
        """Подключение почтового аккаунта"""
        if account_index >= len(self.config['email_accounts']):
            console.print("❌Индекс учетной записи почтового ящика вне диапазона", style="red")
            return False
        
        account_config = self.config['email_accounts'][account_index]
        self.connector = EmailConnector(account_config)
        
        return self.connector.connect_imap()
    
    def process_emails(self, limit=20):
        """Основной процесс обработки почты"""
        if not self.connector:
            console.print("❌ПожалуйстаПодключение почтового аккаунта", style="red")
            return
        
        console.print("🚀Начать обработку сообщений...", style="blue")
        
        # Получить почту        emails = self.connector.get_emails(limit=limit)
        self.processing_stats['total_emails'] = len(emails)
        
        if not emails:
            console.print("📭Сообщений не найдено", style="yellow")
            return
        
        console.print(f"📧Найдено:{len(emails)}письма для начала обработки...", style="green")
        
        # Обработать каждое сообщение        for email_data in tqdm(emails, desc="- Работа с почтой"):
            try:
                processed_email = self.process_single_email(email_data)
                self.processed_emails.append(processed_email)
            except Exception as e:
                console.print(f"❌Не удалось обработать сообщение: {str(e)}", style="red")
                continue
        
        console.print("✅Сообщение обработано!", style="green")
        self.display_processing_summary()
    
    def process_single_email(self, email_data):
        """Обработка одного письма"""
        # 1.Классификация сообщений        classification = self.classifier.classify_email(email_data)
        self.processing_stats['classified_emails'] += 1
        
        # 2.Извлечение информации        extracted_info = self.extractor.generate_summary(email_data)
        
        # 3.Сгенерировать черновик ответа        reply_draft = None
        if classification['type'] != 'spam':  #Ответ на спам не сгенерирован            reply_draft = self.reply_generator.generate_reply(email_data, classification)
            self.processing_stats['replies_generated'] += 1
        
        # 4.Создать напоминание        reminders = []
        if classification['priority'] in ['high', 'medium']:
            reminders = self.reminder_manager.create_reminders(email_data, extracted_info)
            self.processing_stats['reminders_created'] += len(reminders)
        
        # Результаты обработки сборки        processed_email = {
            'original_email': email_data,
            'classification': classification,
            'extracted_info': extracted_info,
            'reply_draft': reply_draft,
            'reminders': reminders,
            'processed_at': datetime.now().isoformat()
        }
        
        return processed_email
    
    def display_processing_summary(self):
        """Отображение сводки обработки"""
        table = Table(title="📊Сводка по обработке сообщений")
        table.add_column("Проект «Строительство ", style="cyan")
        table.add_column("Кол-во", style="magenta")
        
        table.add_row("Всего писем", str(self.processing_stats['total_emails']))
        table.add_row("Классифицировано", str(self.processing_stats['classified_emails']))
        table.add_row("Генерация черновика ответа", str(self.processing_stats['replies_generated']))
        table.add_row("Создание напоминания", str(self.processing_stats['reminders_created']))
        
        console.print(table)
    
    def get_classification_stats(self):
        """Получение статистики классификации"""
        if not self.processed_emails:
            return {}
        
        stats = {
            'type': {},
            'priority': {},
            'sender_type': {}
        }
        
        for email in self.processed_emails:
            classification = email['classification']
            
            # Тип статистики            email_type = classification['type']
            stats['type'][email_type] = stats['type'].get(email_type, 0) + 1
            
            # Статистический приоритет            priority = classification['priority']
            stats['priority'][priority] = stats['priority'].get(priority, 0) + 1
            
            # Тип отправителя статистики            sender_type = classification['sender_type']
            stats['sender_type'][sender_type] = stats['sender_type'].get(sender_type, 0) + 1
        
        return stats
    
    def save_results(self, output_dir='output'):
        """Сохранение результатов обработки"""
        import os
        
        # Создать выходной каталог        os.makedirs(f"{output_dir}/reports", exist_ok=True)
        os.makedirs(f"{output_dir}/drafts", exist_ok=True)
        
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        
        # Сохранить отчет об обработке        report_data = {
            'processing_stats': self.processing_stats,
            'classification_stats': self.get_classification_stats(),
            'reminder_summary': self.reminder_manager.get_reminders_summary(),
            'processed_emails': self.processed_emails,
            'generated_at': datetime.now().isoformat()
        }
        
        with open(f"{output_dir}/reports/email_report_{timestamp}.json", 'w', encoding='utf-8') as f:
            json.dump(report_data, f, ensure_ascii=False, indent=2)
        
        # Сохранить черновик ответа        drafts = []
        for email in self.processed_emails:
            if email['reply_draft']:
                drafts.append({
                    'original_subject': email['original_email']['subject'],
                    'original_sender': email['original_email']['sender'],
                    'reply': email['reply_draft']
                })
        
        with open(f"{output_dir}/drafts/reply_drafts_{timestamp}.json", 'w', encoding='utf-8') as f:
            json.dump(drafts, f, ensure_ascii=False, indent=2)
        
        console.print(f"✅Результаты сохранены в{output_dir}СОДЕРЖАНИЕ", style="green")

print("✅ Интеллектуальный почтовый помощникОпределение основной программы завершено")

## 9. Визуализация и генерация отчётов

In [ ]:
def create_visualization(assistant):
    """Создание визуализаций"""
    if not assistant.processed_emails:
        console.print("❌Нет обработанных данных электронной почты", style="red")
        return
    
    stats = assistant.get_classification_stats()
    
    # Создать подграф    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('Отчет об анализе обработки почты', fontsize=16, fontweight='bold')
    
    # 1.Рассылка типов сообщений    if stats['type']:
        type_labels = list(stats['type'].keys())
        type_values = list(stats['type'].values())
        
        axes[0, 0].pie(type_values, labels=type_labels, autopct='%1.1f%%', startangle=90)
        axes[0, 0].set_title('Рассылка типов сообщений')
    
    # 2.Приоритетное распределение    if stats['priority']:
        priority_labels = list(stats['priority'].keys())
        priority_values = list(stats['priority'].values())
        
        colors = {'high': 'red', 'medium': 'orange', 'low': 'green'}
        bar_colors = [colors.get(label, 'blue') for label in priority_labels]
        
        axes[0, 1].bar(priority_labels, priority_values, color=bar_colors)
        axes[0, 1].set_title('Распределение приоритетов сообщений')
        axes[0, 1].set_ylabel('Кол-во')
    
    # 3.Распределение типа отправителя    if stats['sender_type']:
        sender_labels = list(stats['sender_type'].keys())
        sender_values = list(stats['sender_type'].values())
        
        axes[1, 0].bar(sender_labels, sender_values)
        axes[1, 0].set_title('Распределение типа отправителя')
        axes[1, 0].set_ylabel('Кол-во')
        axes[1, 0].tick_params(axis='x', rotation=45)
    
    # 4.Статистика обработки    process_labels = ['Всего', 'Классиф.', 'Ответ', 'Напоминание']
    process_values = [
        assistant.processing_stats['total_emails'],
        assistant.processing_stats['classified_emails'],
        assistant.processing_stats['replies_generated'],
        assistant.processing_stats['reminders_created']
    ]
    
    axes[1, 1].bar(process_labels, process_values, color='skyblue')
    axes[1, 1].set_title('Статистика обработки')
    axes[1, 1].set_ylabel('Кол-во')
    axes[1, 1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()

def display_sample_results(assistant, num_samples=3):
    """Отображение примеров результатов"""
    if not assistant.processed_emails:
        console.print("❌Нет обработанных данных электронной почты", style="red")
        return
    
    console.print("\n📋Результаты обработки образцов:", style="bold blue")
    
    for i, email in enumerate(assistant.processed_emails[:num_samples]):
        console.print(f"\n---Почта{i+1} ---", style="yellow")
        
        # Информация о необработанном сообщении        original = email['original_email']
        console.print(f"УЧЕБНОГО ПРЕДМЕТА: {original['subject']}", style="cyan")
        console.print(f"Отправитель: {original['sender']}", style="cyan")
        
        # Результаты категоризации        classification = email['classification']
        console.print(f"Модель: {classification['type']} |Приоритет: {classification['priority']} |Тип отправителя: {classification['sender_type']}", style="green")
        
        # Извлеченная информация        extracted = email['extracted_info']
        if extracted['key_dates']:
            console.print(f"Ключевые даты: {', '.join(extracted['key_dates'])}", style="magenta")
        if extracted['todo_items']:
            console.print(f"Делать: {extracted['todo_items'][0][:50]}...", style="magenta")
        
        # Проект ответа        if email['reply_draft']:
            reply = email['reply_draft']
            console.print(f"Проект ответа({reply['tone']}, {reply['language']}): {reply['content'][:100]}...", style="white")
        
        # Напоминание        if email['reminders']:
            console.print(f"Создано{len(email['reminders'])}оповещения", style="yellow")

print("✅Определение функции визуализации и отчетности завершено")

## 10. Демонстрация и тестирование

In [ ]:
# Создание демонстрационных данных(Если вы не можете подключиться к своему реальному адресу электронной почты)def create_demo_data():
    """Создание демонстрационных данных"""
    demo_emails = [
        {
            'id': '1',
            'subject': 'Срочно: планирование встречи по статусу проекта',
            'sender': 'manager@company.com',
            'date': '2024-01-15 09:00:00',
            'body': 'Коллеги, подготовьтесь к встрече по статусу проекта завтра в 14:00. Нужны итоги недели и план на следующую. Дедлайн: 2024-01-16 14:00. Подтвердите участие.'
        },
        {
            'id': '2',
            'subject': 'Клиент: детали продукта',
            'sender': 'customer@client.com',
            'date': '2024-01-15 10:30:00',
            'body': 'Здравствуйте, меня интересует ваш продукт. Можно ли назначить демонстрацию? Телефон: 13800138000. Жду ответа.'
        },
        {
            'id': '3',
            'subject': 'Уведомление о техническом обслуживании системы',
            'sender': 'noreply@system.com',
            'date': '2024-01-15 11:00:00',
            'body': 'Обновление 2024-01-20 02:00-04:00. Подготовьтесь. Вопросы — в поддержку.'
        },
        {
            'id': '4',
            'subject': 'Акция! Скидка 20%',
            'sender': 'promotion@ads.com',
            'date': '2024-01-15 12:00:00',
            'body': 'Акция! Скидка 20%. Не упустите!'
        },
        {
            'id': '5',
            'subject': 'Личное: встреча на выходных',
            'sender': 'friend@personal.com',
            'date': '2024-01-15 13:00:00',
            'body': 'Встреча в субботу в 19:00. Подтвердите участие.'
        }
    ]
    
    return demo_emails

def run_demo():
    """Запуск демонстрации"""
    console.print("🎯Перейти в режим презентацииИнтеллектуальный почтовый помощник", style="bold blue")
    
    # Просмотреть конфигурацию    if not config or not templates:
        console.print("❌Не удалось загрузить конфигурацию или шаблон, не удалось запустить демонстрацию", style="red")
        return
    
    # Создать экземпляр помощника    assistant = EmailSmartAssistant(config, templates)
    
    # Работа с демо-данными    console.print("📧Тест с демонстрационными данными...", style="yellow")
    demo_emails = create_demo_data()
    
    # Обработка демонстрационных писем    assistant.processing_stats['total_emails'] = len(demo_emails)
    
    for email_data in tqdm(demo_emails, desc="Обработка демонстрационных писем"):
        try:
            processed_email = assistant.process_single_email(email_data)
            assistant.processed_emails.append(processed_email)
        except Exception as e:
            console.print(f"❌Не удалось обработать сообщение: {str(e)}", style="red")
            continue
    
    # Показать результаты    console.print("\n✅Демо-обработка завершена!", style="green")
    assistant.display_processing_summary()
    
    # Показать результаты выборки    display_sample_results(assistant)
    
    # Создать визуализацию    create_visualization(assistant)
    
    # & Сохранить результат в файл    assistant.save_results()
    
    return assistant

print("✅Подготовка докладчика завершена")

## 11. Запуск интеллектуального почтового помощника

In [ ]:
# Запустить демоassistant = run_demo()

## 12. Подключение реального почтового ящика (опционально)

In [ ]:
# Если вы хотите подключиться к реальному почтовому ящику, сначала настройте конфигурацию/email_config.json-файл# Затем раскомментируйте код ниже
# def run_with_real_email():
# """Запуск с реальным почтовым ящиком"""
# console.print("🔗Подключиться к реальному электронному письму...", style="blue")
# # #Создать экземпляр помощника# assistant = EmailSmartAssistant(config, templates)
# # #Подключить электронную почту# if not assistant.connect_email_account(0):  #Использовать первую учетную запись электронной почты# console.print("❌Не удалось подключиться к почтовому ящику, style="red")
# return None
# # #- Работа с почтой# assistant.process_emails(limit=10)  #Обработка последних 10 писем# # #Показать результаты# display_sample_results(assistant)
# create_visualization(assistant)
# assistant.save_results()
# # # Закрытие соединения
# assistant.connector.close_connections()
# # return assistant

# #Запуск реальной обработки почтовых ящиков# real_assistant = run_with_real_email()

console.print("\n🎉 Интеллектуальный почтовый помощникДемо завершено!", style="bold green")
console.print("\n📝Инструкция по применению :", style="bold yellow")
console.print("1.Изменить конфигурацию/email_config.json настраивает информацию о вашей электронной почте")
console.print("2.Раскомментировать реальный код подключения к электронной почте выше")
console.print("3.Запустите код, чтобы начать обработку почты")
console.print("4.Просмотр отчетов об обработке и черновиков ответов в выходном каталоге")